# 🦷 Teeth Segmentation with U-Net (ResNet50 Encoder)

A binary segmentation model that takes a photo of teeth and predicts a pixel-level mask of the teeth region.

- **Dataset:** self-collected images with hand-drawn (manually annotated) masks — 291 image/mask pairs.
- **Architecture:** U-Net decoder on top of a **ResNet50** encoder pretrained on ImageNet (encoder frozen — transfer learning), with skip connections taken from the ResNet50 feature maps.
- **Loss / metric:** Dice loss and Dice coefficient (standard choice for imbalanced binary segmentation, where the foreground — teeth — is a small fraction of the image).
- **Output:** a 256×256 binary mask (sigmoid activation, thresholded at 0.5).

## Setup & Dataset

Unzip the dataset (images + hand-annotated masks) in the Colab environment.

In [ ]:
!pip install opencv-python matplotlib scikit-learn tensorflow

In [ ]:
!unzip Dataset-u-net.zip -d /content/

In [ ]:
!ls /content/

In [ ]:
!ls /content/Dataset-u-net

## Load & Preprocess Data

Loads each image and its corresponding hand-annotated mask, resizes both to 256×256, normalizes the image to [0, 1], and binarizes the mask (threshold 0.5).

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

def load_teeth_data(images_dir, masks_dir, img_size=(256, 256)):
    images = []
    masks = []

    # Fetch and sort filenames from the two folders independently
    image_names = sorted(os.listdir(images_dir))
    mask_names = sorted(os.listdir(masks_dir))

    print(f"Files found in the images folder: {len(image_names)}")
    print(f"Files found in the masks folder: {len(mask_names)}")

    if len(image_names) != len(mask_names):
        print("⚠️ Warning: number of images doesn't match number of masks! Reading will proceed by matching order.")

    # Read by matching order (entry i pairs with entry i) to avoid filename mismatches
    for i in range(min(len(image_names), len(mask_names))):
        img_name = image_names[i]
        mask_name = mask_names[i]

        # 1. Read and prepare the original image
        img_path = os.path.join(images_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)

        # 2. Read and prepare the corresponding mask
        mask_path = os.path.join(masks_dir, mask_name)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Verify both files were read successfully
        if img is None:
            print(f"❌ Failed to read image: {img_name}")
            continue
        if mask is None:
            print(f"❌ Failed to read mask: {mask_name}")
            continue

        # Process the image
        img = cv2.resize(img, img_size)
        img = img / 255.0

        # Process the mask
        mask = cv2.resize(mask, img_size)
        mask = mask / 255.0
        mask = np.where(mask > 0.5, 1.0, 0.0)
        mask = np.expand_dims(mask, axis=-1)

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# Dataset paths in Colab
IMAGES_PATH = '/content/Dataset-u-net/Img'
MASKS_PATH = '/content/Dataset-u-net/masks_output'

# Run the loading
X, y = load_teeth_data(IMAGES_PATH, MASKS_PATH)

if X.shape[0] > 0:
    print(f"\n✅ Successfully loaded {X.shape[0]} images.")
    print("Images array shape:", X.shape)
    print("Masks array shape:", y.shape)

    # Split the data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set size: {X_train.shape[0]} | Validation set size: {X_val.shape[0]}")
else:
    print("\n❌ The array is still empty. Please make sure the folders contain actual image files and not other subfolders.")


## Model: ResNet50-U-Net Architecture

A U-Net decoder built on top of a frozen, ImageNet-pretrained ResNet50 encoder. Skip connections are taken from four ResNet50 feature maps at different resolutions (`conv1_relu`, `conv2_block3_out`, `conv3_block4_out`) plus the bottleneck (`conv4_block6_out`), and concatenated into the decoder path — the standard U-Net pattern, adapted to reuse ResNet50's pretrained features instead of training an encoder from scratch.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_resnet50_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)

    # 1. Load the pretrained ResNet50
    encoder = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )

    # Freeze the encoder to preserve the fine-grained ImageNet features
    encoder.trainable = False

    # 2. Extract the skip connections (the first skip uses `inputs` directly
    # to avoid a layer-naming mismatch)
    s1 = inputs                                                   # (256, 256, 3)
    s2 = encoder.get_layer("conv1_relu").output                   # (128, 128, 64)
    s3 = encoder.get_layer("conv2_block3_out").output             # (64, 64, 256)
    s4 = encoder.get_layer("conv3_block4_out").output             # (32, 32, 512)

    # Bottleneck
    bridge = encoder.get_layer("conv4_block6_out").output          # (16, 16, 1024)

    # 3. Build the decoder (the right half of the U-Net)
    # Block 1 (upsample to 32x32, connect to s4)
    u1 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(bridge)
    u1 = layers.concatenate([u1, s4])
    c1 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u1)
    c1 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c1)

    # Block 2 (upsample to 64x64, connect to s3)
    u2 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c1)
    u2 = layers.concatenate([u2, s3])
    c2 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u2)
    c2 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c2)

    # Block 3 (upsample to 128x128, connect to s2)
    u3 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c2)
    u3 = layers.concatenate([u3, s2])
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u3)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)

    # Block 4 (upsample to 256x256, connect to s1, the original input)
    u4 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c3)
    u4 = layers.concatenate([u4, s1])
    c4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    c4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c4)

    # Final layer producing the binary teeth mask
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c4)

    model = Model(inputs, outputs, name="ResNet50_UNet")
    return model

# Build and compile the model
model = build_resnet50_unet(input_shape=(256, 256, 3))

# Dice coefficient & Dice loss
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=dice_loss,
    metrics=[dice_coef, 'accuracy']
)

print("✅ Model built and compiled successfully!")


## Training

Trains for 20 epochs, checkpointing the best weights (by `val_loss`) to `best_teeth_resnet_unet.h5`.

In [ ]:
# Save the best weights automatically whenever val_loss improves
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_teeth_resnet_unet.h5',
        save_best_only=True,
        monitor='val_loss',
        mode='min'
    )
]

print("🚀 Starting model training on teeth images...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,        # can be increased later (e.g. 40-50) to improve accuracy
    batch_size=8,     # lower this if you hit an Out of Memory error
    callbacks=callbacks
)


## Save the Trained Model

In [ ]:
model.save('full_teeth_segmentation_model.h5')

## Inference on a New Image

Loads the saved model, lets you upload a new teeth photo (via Colab's file picker), and displays the predicted mask next to the original image.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import files
from google.colab import drive

drive.mount('/content/drive')

try:
    model = tf.keras.models.load_model('/content/drive/MyDrive/full_teeth_segmentation_model.h5', compile=False)
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Error loading model: {e}")

print("📸 Please choose a teeth image to upload:")
uploaded = files.upload()

if len(uploaded) > 0:
    img_name = list(uploaded.keys())[0]
    new_img = cv2.imread(img_name, cv2.IMREAD_COLOR)

    if new_img is not None:
        display_img = cv2.cvtColor(new_img, cv2.COLOR_BGR2RGB)
        resized_img = cv2.resize(new_img, (256, 256))
        normalized_img = resized_img / 255.0
        input_tensor = np.expand_dims(normalized_img, axis=0)

        predicted_mask = model.predict(input_tensor)[0]
        predicted_mask = np.where(predicted_mask > 0.5, 1.0, 0.0)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title("Uploaded Image")
        plt.imshow(display_img)
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.title("Predicted Mask")
        plt.imshow(predicted_mask.squeeze(), cmap='gray')
        plt.axis('off')
        plt.show()
    else:
        print("❌ Could not read the uploaded image.")
else:
    print("❌ No image was uploaded.")
